### Retrieval-Augmented Generation (RAG) Chatbot

This notebook provides a high-level, end-to-end overview of how to build a RAG chatbot. A RAG chatbot connects a large language model (LLM) to data sources like PDFs and websites, allowing the chatbot to give context-aware and accurate answers to frequently-asked questions.

**WARNING:** This is my very first attempt, so please don't judge too harshly! If you learned something new from this notebook, or have suggestions for improvement, I would love to hear from you.

### Prepare Data

- Scrape text from the UW-Madison DAPIR website.
- Clean text to remove whitespace and special characters.
- Chunk text into smaller, semantically-meaningful units (500 words).

In [1]:
# Import libraries
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
from typing import List, Dict
import re

# Specify constants (URL and max links to scrape)
# Start small for testing, setting MAX_LINKS to 3
BASE_URL = "https://data.wisc.edu/"
MAX_LINKS = 3

In [2]:
def fetch_webpage(url: str) -> BeautifulSoup:
    """
    Fetch a webpage and return a BeautifulSoup object (parsed HTML).
    """
    # Use try-except for error handling
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        return soup # Return parsed HTML
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}") # Print error for debugging
        return None


def extract_links(soup: BeautifulSoup, base_url: str) -> List[str]:
    """
    Extract all links from the base url and convert them to absolute URLs.
    """
    # Initialize container for links
    links = []
    
    # Loop through all anchors (the <a> HTML element which defines hyperlinks)
    for anchor in soup.find_all('a', href=True):
        href = anchor['href']
        
        # Convert to absolute URL
        absolute_url = urljoin(base_url, href)

        # Include only links within the same domain
        if urlparse(absolute_url).netloc != urlparse(base_url).netloc:
            continue
        
        # Remove non-HTML content like PDFs and images
        # TODO: Are there ways to use non-HTML content for RAG context? Multi-modal RAG?
        if not any(absolute_url.endswith(ext) for ext in ['.pdf', '.jpg', '.png', '.zip']):
            if '#' not in absolute_url or absolute_url.split('#')[0] not in links:
                links.append(absolute_url.split('#')[0])
    
    # Remove duplicate links because a single page may link to the same URL multiple times
    seen = set()
    unique_links = []
    for link in links:
        if link not in seen:
            seen.add(link)
            unique_links.append(link)
    
    return unique_links

# Fetch the main webpage, extract all unique links, and subset links
main_soup = fetch_webpage(BASE_URL)
all_links = extract_links(main_soup, BASE_URL)
selected_links = all_links[:MAX_LINKS]

# Display selected links
print(f"Selected {len(selected_links)} links for RAG Chatbot context:")
for i, link in enumerate(selected_links, 1):
    print(f"  {i}. {link}")

Selected 3 links for RAG Chatbot context:
  1. https://data.wisc.edu/
  2. https://data.wisc.edu/about-us/
  3. https://data.wisc.edu/dapir-staff/


In [3]:
def extract_text_content(soup: BeautifulSoup, url: str) -> Dict[str, str]:
    """
    Extract meaningful text content from a url.
    """
    # Remove script, style, and navigation elements
    for element in soup(['script', 'style', 'nav', 'footer', 'header', 'iframe']):
        element.decompose()
    
    # Extract title
    title = soup.find('title')
    title_text = title.get_text().strip() if title else "No Title"
    
    # Extract main content
    # Try to find main content area (common patterns)
    main_content = (
        soup.find('main') or 
        soup.find('article') or 
        soup.find('div', {'class': ['content', 'main-content', 'page-content']}) or
        soup.find('body')
    )
    
    # Extract text from headings (h1, h2, h3), paragraphs (p) and lists (li)
    text_elements = []
    
    if main_content:
        for tag in main_content.find_all(['h1', 'h2', 'h3', 'h4', 'p', 'li']):
            text = tag.get_text().strip()
            if text and len(text) > 20:  # Filter out very short snippets
                text_elements.append(text)
    
    combined_text = '\n\n'.join(text_elements)
    
    return {
        'url': url,
        'title': title_text,
        'content': combined_text,
        'word_count': len(combined_text.split())
    }


# Call functions to scrape content from selected links
scraped_data = []

for i, url in enumerate(selected_links, 1):
    
    # Fetch the page
    print(f"[{i}/{len(selected_links)}] Scraping: {url}")
    soup = fetch_webpage(url)
    
    # Extract content
    page_data = extract_text_content(soup, url)
    
    # Display results
    if page_data and page_data['word_count'] > 0:
        scraped_data.append(page_data)
        print(f"Total scraped words: {page_data['word_count']}")
    else:
        print(f"Error: No content extracted")
    
    # Include a sleep buffer because the server may block rapid requests
    time.sleep(1)

[1/3] Scraping: https://data.wisc.edu/
Total scraped words: 313
[2/3] Scraping: https://data.wisc.edu/about-us/
Total scraped words: 285
[3/3] Scraping: https://data.wisc.edu/dapir-staff/
Total scraped words: 184


In [4]:
def clean_text(text: str) -> str:
    """
    Clean the scraped text content to prepare for chunking.
    """
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^\w\s.,!?;:()\-\'\"]+', '', text)
    
    # Remove repeated punctuation
    text = re.sub(r'([.,!?;:]){2,}', r'\1', text)
    
    # Return cleaned text
    return text.strip()

def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> List[Dict[str, any]]:
    """
    Split text into overlapping chunks (500 words each, overlapping by 50 words) to preserve words' context.
    """
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk_words = words[i:i + chunk_size]
        chunk_text = ' '.join(chunk_words)
        
        if len(chunk_words) > 20:  # TODO: Experiment with including smaller chunks too
            chunks.append({
                'text': chunk_text,
                'word_count': len(chunk_words),
                'chunk_index': len(chunks)
            })
    
    return chunks

# Call functions to clean and chunk scraped data
processed_documents = []

for page_data in scraped_data:
    # Clean the content
    cleaned_content = clean_text(page_data['content'])
    
    # Chunk the content
    chunks = chunk_text(cleaned_content, chunk_size=500, overlap=50)
    
    # Add metadata to each chunk
    for chunk in chunks:
        processed_documents.append({
            'source_url': page_data['url'],
            'chunk_text': chunk['text'],
            'chunk_index': chunk['chunk_index'],
            'word_count': chunk['word_count']
        })
    
    # Display results
    print(f"Processed {page_data['url']} and created {len(chunks)} chunks")

Processed https://data.wisc.edu/ and created 1 chunks
Processed https://data.wisc.edu/about-us/ and created 1 chunks
Processed https://data.wisc.edu/dapir-staff/ and created 1 chunks


### Convert and Store Embeddings

- Convert text chunks into vector embeddings with a pre-trained model (HuggingFace sentence-transformers).
- Store embeddings and metadata in an open-source vector database (ChromaDB) to index chunks with embeddings and metadata like URLs.
- Whereas traditional databases use exact matching, vector databases use similarity search to find relevant content.

In [5]:
# Import libraries
from sentence_transformers import SentenceTransformer
import numpy as np

# Load pre-trained embeddings model from HuggingFace
# TODO: Experiment with different (free) models
model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract text to generate embeddings for each chunk
chunk_texts = [doc['chunk_text'] for doc in processed_documents]

# Generate embeddings
embeddings = model.encode(
    chunk_texts,
    batch_size=32, # batch to process efficiently
    convert_to_numpy=True
)

# Add embeddings to processed documents
for i, doc in enumerate(processed_documents):
    doc['embedding'] = embeddings[i]

# Display results
print(f"Success! Embeddings added to documents.")
print(f"Example chunk text: {processed_documents[0]['chunk_text'][:100]}...")
print(f"Example embedding: [{processed_documents[0]['embedding'][:5]}...]")

Success! Embeddings added to documents.
Example chunk text: Join us February 18 at 10 am via Zoom for the Badger Data Network The Center for User Experience (CU...
Example embedding: [[ 0.05567217 -0.06895624 -0.01204216  0.01153835  0.01390631]...]


In [6]:
# Import libraries
import chromadb
from chromadb.config import Settings

# Initialize ChromaDB client with persistent storage to save and reuse data
client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=Settings(
        anonymized_telemetry=False,  # Disable telemetry to protect data privacy
        allow_reset=True # For development only! Do NOT allow in production because it can lead to accidental data loss!
    )
)

# Create or get a collection of related documents (like a table in SQL)
collection_name = "DAPIR_website_chunks"

# Delete existing collection if it exists (for fresh start during development)
try:
    client.delete_collection(name=collection_name)
except:
    pass

# Create new collection
collection = client.create_collection(
    name=collection_name,
    metadata={
        "description": "DAPIR website content chunks with embeddings",
        "embedding_model": "all-MiniLM-L6-v2",
        "created_date": time.strftime("%Y-%m-%d %H:%M:%S") # Store creation date
    }
)

# Prepare data for ChromaDB
# ChromaDB expects: ids, documents, embeddings, metadatas

ids = [f"chunk_{i}" for i in range(len(processed_documents))]
documents = [doc['chunk_text'] for doc in processed_documents]
embeddings_list = [doc['embedding'].tolist() for doc in processed_documents]

metadatas = [{
    'source_url': doc['source_url'],
    'chunk_index': doc['chunk_index'],
    'word_count': doc['word_count']
} for doc in processed_documents]

# Add documents to ChromaDB
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings_list,
    metadatas=metadatas
)

### Retrieval

- Convert user prompts into embeddings using the same, pre-trained model.
- Execute similarity search to find the most relevant text chunks.

In [ ]:
def retrieve_relevant_chunks(
    query: str, 
    collection, 
    model, 
    n_results: int = 5
) -> Dict[str, any]:
    """
    Retrieve the most relevant text chunks for a user query.
    
    This is the core retrieval function for the RAG system:
    1. Convert the user's question to an embedding
    2. Search the vector database for similar embeddings
    3. Return the most relevant text chunks with metadata
    
    Args:
        query: User's question or search query
        collection: ChromaDB collection containing document embeddings
        model: Sentence-transformer model (same one used for embedding documents)
        n_results: Number of relevant chunks to retrieve (default: 5)
        
    Returns:
        Dictionary containing documents, metadata, and similarity scores
    """
    # Step 1: Convert query to embedding using the same model
    print(f"Query: '{query}'")
    print(f"\nConverting query to embedding...")
    
    query_embedding = model.encode([query]).tolist()
    
    # Step 2: Search vector database for similar chunks
    print(f"Searching for {n_results} most relevant chunks...")
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )
    
    # Step 3: Format and return results
    print(f"✓ Found {len(results['documents'][0])} relevant chunks\n")
    
    return results


def display_retrieval_results(query: str, results: Dict[str, any], show_full_text: bool = False):
    """
    Display retrieval results in a user-friendly format.
    
    Args:
        query: The original query
        results: Results from retrieve_relevant_chunks()
        show_full_text: Whether to show full chunk text (default: False, shows preview)
    """
    print(f"{'='*80}")
    print(f"RETRIEVAL RESULTS")
    print(f"{'='*80}")
    print(f"Query: '{query}'")
    print(f"Retrieved {len(results['documents'][0])} chunks\n")
    
    for i, (doc, metadata, distance) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ), 1):
        # Convert distance to similarity score (0-1, higher is better)
        similarity = 1 - distance
        
        print(f"{'-'*80}")
        print(f"CHUNK #{i} | Similarity: {similarity:.4f}")
        print(f"{'-'*80}")
        print(f"Source Page: {metadata['page_title']}")
        print(f"URL: {metadata['source_url']}")
        print(f"Chunk Index: {metadata['chunk_index']} | Words: {metadata['word_count']}")
        print(f"\nContent:")
        
        if show_full_text:
            print(doc)
        else:
            # Show first 300 characters as preview
            preview = doc[:300] + "..." if len(doc) > 300 else doc
            print(preview)
        print()
    
    print(f"{'='*80}\n")


# ============================================================================
# TEST RETRIEVAL SYSTEM
# ============================================================================

# Test with multiple queries
test_queries = [
    "What does DAPIR manage?",
    "How can I contact DAPIR?",
    "Can DAPIR help with Tableau?"
]

print("Testing Retrieval System with Sample Queries")
print(f"{'='*80}\n")

for test_query in test_queries:
    # Retrieve relevant chunks
    results = retrieve_relevant_chunks(
        query=test_query,
        collection=collection,
        model=model,
        n_results=3  # Get top 3 most relevant chunks
    )
    
    # Display results
    display_retrieval_results(test_query, results, show_full_text=False)
    
    # Separator between queries
    print("\n" + "█"*80 + "\n")

print(f"\n{'='*80}")
print("✓ Retrieval system is working!")
print(f"{'='*80}")
print("\nNOTE: These chunks will be passed to the LLM in the next step")
print("      to generate context-aware answers to user questions.")

### Generation

- Pass the retrieved chunks as **context** to an open-source LLM.
- LLM prompts = user prompt + retrieved context + sys. instructions
- Generate an answer grounded in the source documents.
- Optionally include references to source documents.

In [ ]:
# Import libraries
from transformers import pipeline
import torch

# Check if GPU is available
device = 0 if torch.cuda.is_available() else -1
device_name = "GPU" if device == 0 else "CPU"

print(f"Using device: {device_name}")

# Initialize a lightweight text generation pipeline
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base", # Local model for RAG
    device=device,
    max_length=512
)

def format_context_with_sources(results: Dict[str, any]) -> tuple:
    """
    Format retrieved chunks into context text and extract unique sources.
    
    Args:
        results: Results from retrieve_relevant_chunks()
        
    Returns:
        Tuple of (context_text, sources_list)
    """
    context_parts = []
    sources = {}  # Use dict to track unique sources with titles
    
    for i, (doc, metadata) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0]
    ), 1):
        # Add context with source reference
        context_parts.append(f"[Source {i}]: {doc}")
        
        # Track unique sources
        url = metadata['source_url']
        if url not in sources:
            sources[url] = metadata['page_title']
    
    context_text = "\n\n".join(context_parts)
    sources_list = [{"url": url, "title": title} for url, title in sources.items()]
    
    return context_text, sources_list


def generate_answer(
    query: str,
    context: str,
    sources: List[Dict],
    max_tokens: int = 200
) -> str:
    """
    Generate an answer using retrieved context and local FLAN-T5 model.
    
    Args:
        query: User's question
        context: Retrieved text chunks formatted as context
        sources: List of source URLs and titles
        max_tokens: Maximum length of generated answer
        
    Returns:
        Generated answer with source citations
    """
    # Truncate context if too long (FLAN-T5 has 512 token input limit)
    max_context_length = 800
    if len(context) > max_context_length:
        context = context[:max_context_length] + "..."
    
    # Build sources list
    sources_text = "\n".join([f"- {s['title']}: {s['url']}" for s in sources])
    
    # Create prompt optimized for FLAN-T5
    # FLAN-T5 works best with direct, clear instructions
    prompt = f"""Answer the question based only on the context below.

Context: {context}

Question: {query}

Answer:"""
    
    # Generate response
    try:
        print("   Generating answer with local model...")
        
        result = generator(
            prompt,
            max_length=max_tokens,
            do_sample=True,
            temperature=0.4,
            top_p=0.9
        )
        
        answer = result[0]['generated_text'].strip()
        
        if not answer:
            return "I couldn't generate an answer based on the provided context."
        
        # Add source citations
        answer += f"\n\nSources:\n{sources_text}"
        
        return answer
        
    except Exception as e:
        error_msg = str(e)
        return f"❌ Error generating response: {error_msg}"


def rag_chatbot(query: str, n_results: int = 5, show_context: bool = False):
    """
    Complete RAG pipeline: Retrieve context and generate answer.
    
    Args:
        query: User's question
        n_results: Number of chunks to retrieve
        show_context: Whether to display retrieved context
    """
    print(f"{'='*80}")
    print(f"RAG CHATBOT")
    print(f"{'='*80}")
    print(f"Question: {query}\n")
    
    # Step 1: Retrieve relevant chunks
    print("Step 1: Retrieving relevant context...")
    results = retrieve_relevant_chunks(
        query=query,
        collection=collection,
        model=model,  # This is the sentence-transformers embedding model
        n_results=n_results
    )
    
    # Step 2: Format context and extract sources
    context, sources = format_context_with_sources(results)
    
    if show_context:
        print(f"\n{'─'*80}")
        print("RETRIEVED CONTEXT:")
        print(f"{'─'*80}")
        print(context[:500] + "...\n" if len(context) > 500 else context + "\n")
    
    # Step 3: Generate answer
    print("Step 2: Generating answer...")
    answer = generate_answer(
        query=query,
        context=context,
        sources=sources,
        max_tokens=200
    )
    
    # Display answer
    print(f"\n{'─'*80}")
    print("ANSWER:")
    print(f"{'─'*80}")
    print(answer)
    print(f"{'='*80}\n")

### Model Evaluation

- Test the chatbot with sample FAQs.
- Are the answers accurate and helpful?

In [ ]:
print("\nTesting Complete RAG Chatbot with Q&A\n")

# Test questions
test_questions = [
    "What does DAPIR do?",
    "How can I get help with Tableau?"
]

for question in test_questions:
    rag_chatbot(question, n_results=1, show_context=False)
    print("\n")